In [1]:
import torch

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

mps


In [2]:
import os

from data.augment import AudioAugmenter, AugmentConfig
from data.load_data import (
    TwitDataset,
    collate_fn,
    get_class_imbalance,
    make_augmenting_collate,
)
from torch.utils.data import DataLoader

SEED = 42
torch.manual_seed(SEED)

ds = TwitDataset()  # resamples once to an on-disk cache; later runs just read the cache

# Fixed split: the seed is also stored in the Trainer metadata/checkpoint.
split_generator = torch.Generator().manual_seed(SEED)
train_ds, test_ds = torch.utils.data.random_split(
    ds,
    [0.9, 0.1],
    generator=split_generator,
)

# Preserve a 50% clean path and soften each transform when augmentation is selected.
augment_config = AugmentConfig(
    sample_rate=16_000,
    apply_prob=0.50,
    time_shift_prob=0.35,
    background_mix_prob=0.35,
    background_snr_db=(5.0, 20.0),
    pink_noise_prob=0.25,
    pink_noise_snr_db=(10.0, 30.0),
    level_dbfs=(-45.0, -20.0),
)
train_collate = make_augmenting_collate(AudioAugmenter(augment_config))

NUM_WORKERS = min(4, os.cpu_count() or 1)
loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0,
    pin_memory=(device.type == "cuda"),
)
train_dataloader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=train_collate,
    **loader_kwargs,
)
test_dataloader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    **loader_kwargs,
)

pos_weight = get_class_imbalance(train_ds).to(device)

In [3]:
from models import CNNHead, DSCNNHead, GaborFilter, GaborNet, SpecAugment
from torch import nn

N_FILTERS = 40
SAMPLE_RATE = 16000     # must match the (resampled) audio fed to the model
KERNEL_SIZE = 401       # ~25 ms @ 16 kHz: long enough to resolve ~1 kHz carriers
STRIDE = 160

# Strategy A: fixed, log-spaced constant-Q filterbank (1-8 kHz, Q=12, 40 filters
# -> ~0.66-bandwidth spacing, healthy overlap). Freeze it so the front-end can't
# thrash; only the head learns.
feat_extract = GaborFilter(n_filters=N_FILTERS, kernel_size=KERNEL_SIZE, sample_rate=SAMPLE_RATE, stride=STRIDE)
feat_extract.center_freq.requires_grad_(False)
head = DSCNNHead(channels=(16, 32, 32), activation=nn.LeakyReLU)
model = GaborNet(feat_extract, head).to(device)
model.spec_augment = SpecAugment(
    freq_masks=1,
    time_masks=1,
    max_freq_fraction=0.15,
    max_time_fraction=0.10,
    apply_prob=0.50,
).to(device)

In [4]:
from train import Trainer

trainer = Trainer(
    model,
    train_dataloader,
    test_dataloader,
    pos_weight,
    device,
    lr=1e-2,
    lr_scheduler_kwargs={"eta_min": 1e-4},
    seed=SEED
)
# trainer.train_with_slimming(
#     sparsify_epochs=15,
#     finetune_epochs=15,
#     pruning_ratio=0.32,
#     reg=1e-5,
# )
trainer.train(epochs=35)


  0%|          | 0/35 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

...

  0%|          | 0/112 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

  0%|          | 0/1004 [00:00<?, ?it/s]

  0%|          | 0/112 [00:00<?, ?it/s]

In [5]:
trainer.save_model()

PosixPath('checkpoints/frozen-gabor.pt')